In [ ]:
"""
Incremental market-impact news fetcher – tailored for your Nifty‑50‑50 project.
Appends to your existing news_gap_*.csv file.
"""

import os
import re
import csv
import html
import time
import json
import calendar
from pathlib import Path
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
import logging
logging.basicConfig(level=logging.DEBUG)
import pandas as pd
import requests
from dotenv import load_dotenv

# ----------------------------------------------------------------------
# 1. CONFIGURATION – ADJUSTED TO YOUR EXISTING FOLDER
# ----------------------------------------------------------------------
# Your project base directory
BASE_DIR = Path("c:/Users/laves/backup/nifty-50-50")
NEWS_DIR = BASE_DIR / "news_data"
NEWS_DIR.mkdir(parents=True, exist_ok=True)

# Load environment variables from .env file inside nifty-50-50
load_dotenv(BASE_DIR / ".env", override=True)

# File that stores the date of the last successful fetch
LAST_RUN_FILE = NEWS_DIR / "last_run.json"

def get_latest_csv():
    """
    Find the most recent news_gap_*.csv file in NEWS_DIR.
    If none exists, create a filename with the same pattern using today's date.
    """
    csv_files = list(NEWS_DIR.glob("news_gap_*.csv"))
    if not csv_files:
        today_str = date.today().strftime("%Y_%m_%d")
        return NEWS_DIR / f"news_gap_2025_06_12_to_{today_str}.csv"
    return max(csv_files, key=lambda f: f.stat().st_mtime)

OUTPUT_FILE = get_latest_csv()
print(f"Will append new articles to: {OUTPUT_FILE}")

# API keys
GUARDIAN_KEY = os.getenv("GUARDIAN_KEY", "").strip()
NEWSAPI_KEY  = os.getenv("NEWSAPI_KEY", "").strip()
GNEWS_KEY    = os.getenv("GNEWS_KEY", "").strip()

# Search queries
QUERY_TERMS = [
    "India stock market",
    "Nifty 50",
    "Sensex",
    "NSE India",
    "Indian economy stocks",
    "RBI policy India",
    "Indian corporate earnings",
]

# API limits
REQUEST_TIMEOUT = 30
GUARDIAN_MAX_PAGES_PER_QUERY_MONTH = 5
NEWSAPI_MAX_PAGES_PER_QUERY = 3

# Impact keywords
IMPACT_KEYWORDS = [
    'rbi', 'rate hike', 'rate cut', 'inflation', 'gdp', 'fed', 'recession',
    'earnings', 'profit', 'loss', 'revenue', 'guidance', 'dividend',
    'acquisition', 'merger', 'takeover', 'ipo', 'listing',
    'crash', 'rally', 'record high', 'all-time high', 'plunge', 'sell-off',
    'policy', 'budget', 'deficit', 'fiscal', 'rupee', 'dollar',
    'sensex', 'nifty', 'nse', 'bse', 'sebi', 'morgan stanley', 'goldman',
]
MIN_IMPACT_SCORE = 2

# ----------------------------------------------------------------------
# 2. Helper functions
# ----------------------------------------------------------------------
def clean_text(value):
    """Remove HTML, unescape, collapse whitespace."""
    value = "" if value is None else str(value)
    value = html.unescape(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def month_windows(start_iso, end_iso):
    """Yield (from_date, to_date) tuples for each month in the range."""
    start = date.fromisoformat(start_iso)
    end = date.fromisoformat(end_iso)
    cur = start
    while cur <= end:
        last_day = calendar.monthrange(cur.year, cur.month)[1]
        month_end = date(cur.year, cur.month, last_day)
        window_end = min(month_end, end)
        yield cur.isoformat(), window_end.isoformat()
        cur = window_end + timedelta(days=1)

def compute_impact_score(headline, description):
    """Count how many impact keywords appear in the concatenated text."""
    text = f"{headline} {description}".lower()
    score = sum(1 for kw in IMPACT_KEYWORDS if kw in text)
    return score

def make_row(source, published_iso, title, description, url, query,
             start_date, end_date):
    """
    Build a row dict if the article is within our target date range.
    Also adds impact_score.
    """
    if not published_iso:
        return None

    pub_date_str = published_iso[:10]
    try:
        pub_date = date.fromisoformat(pub_date_str)
    except ValueError:
        return None

    if pub_date < start_date or pub_date > end_date:
        return None

    title_clean = clean_text(title)
    desc_clean = clean_text(description)
    url_clean = clean_text(url)

    if not title_clean or not url_clean:
        return None

    impact_score = compute_impact_score(title_clean, desc_clean)

    return {
        "Archive": source,
        "Date": pub_date.strftime("%d-%m-%Y"),
        "Headline": title_clean,
        "Description": desc_clean if desc_clean else title_clean,
        "Headline link": url_clean,
        "Source": source,
        "Query": query,
        "impact_score": impact_score,
    }

# ----------------------------------------------------------------------
# 3. Last run date management
# ----------------------------------------------------------------------
def get_last_run_date():
    """Read last run date from JSON file, or default to 2025-06-12."""
    if LAST_RUN_FILE.exists():
        with open(LAST_RUN_FILE, 'r') as f:
            data = json.load(f)
            return date.fromisoformat(data["last_run"])
    return date(2025, 6, 12)

def set_last_run_date(run_date):
    """Write last run date to JSON."""
    with open(LAST_RUN_FILE, 'w') as f:
        json.dump({"last_run": run_date.isoformat()}, f)

# ----------------------------------------------------------------------
# 4. API fetching functions
# ----------------------------------------------------------------------
def fetch_guardian(from_date, to_date):
    if not GUARDIAN_KEY:
        print("Skipping Guardian: missing key")
        return []
    rows = []
    endpoint = "https://content.guardianapis.com/search"
    start_d = date.fromisoformat(from_date)
    end_d   = date.fromisoformat(to_date)
    for query in QUERY_TERMS:
        for win_start, win_end in month_windows(from_date, to_date):
            page = 1
            while page <= GUARDIAN_MAX_PAGES_PER_QUERY_MONTH:
                params = {
                    "q": query,
                    "from-date": win_start,
                    "to-date": win_end,
                    "api-key": GUARDIAN_KEY,
                    "show-fields": "headline,trailText,shortUrl",
                    "page-size": 200,
                    "order-by": "newest",
                    "page": page,
                }
                try:
                    resp = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
                except requests.RequestException as e:
                    print(f"Guardian request failed: {e}")
                    break
                if resp.status_code == 429:
                    print("Guardian rate limit. Waiting 10 sec...")
                    time.sleep(10)
                    continue
                if resp.status_code != 200:
                    print(f"Guardian error {resp.status_code}: {query} {win_start} to {win_end}")
                    break
                data = resp.json().get("response", {})
                results = data.get("results", [])
                if not results:
                    break
                for item in results:
                    fields = item.get("fields", {})
                    row = make_row(
                        source="The Guardian",
                        published_iso=item.get("webPublicationDate", ""),
                        title=fields.get("headline") or item.get("webTitle", ""),
                        description=fields.get("trailText", ""),
                        url=fields.get("shortUrl") or item.get("webUrl", ""),
                        query=query,
                        start_date=start_d,
                        end_date=end_d,
                    )
                    if row:
                        rows.append(row)
                total_pages = data.get("pages", 1)
                print(f"Guardian | {query} | {win_start} to {win_end} | page {page}/{min(total_pages, GUARDIAN_MAX_PAGES_PER_QUERY_MONTH)} | rows {len(rows)}")
                if page >= total_pages:
                    break
                page += 1
                time.sleep(0.25)
    return rows


def fetch_newsapi(from_date, to_date):
    if not NEWSAPI_KEY:
        print("Skipping NewsAPI: missing key")
        return []
    rows = []
    endpoint = "https://newsapi.org/v2/everything"
    start_d = date.fromisoformat(from_date)
    end_d   = date.fromisoformat(to_date)
    for query in QUERY_TERMS:
        for page in range(1, NEWSAPI_MAX_PAGES_PER_QUERY + 1):
            params = {
                "q": query,
                "from": from_date,
                "to": to_date,
                "language": "en",
                "sortBy": "publishedAt",
                "pageSize": 100,
                "page": page,
                "apiKey": NEWSAPI_KEY,
            }
            try:
                resp = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
            except requests.RequestException as e:
                print(f"NewsAPI request failed: {e}")
                break
            if resp.status_code != 200:
                msg = resp.json().get("message", resp.text[:120])
                print(f"NewsAPI error {resp.status_code}: {msg}")
                break
            articles = resp.json().get("articles", [])
            if not articles:
                break
            for item in articles:
                row = make_row(
                    source=item.get("source", {}).get("name") or "NewsAPI",
                    published_iso=item.get("publishedAt", ""),
                    title=item.get("title", ""),
                    description=item.get("description", ""),
                    url=item.get("url", ""),
                    query=query,
                    start_date=start_d,
                    end_date=end_d,
                )
                if row:
                    rows.append(row)
            print(f"NewsAPI | {query} | page {page} | rows {len(rows)}")
            time.sleep(0.3)
    return rows


def fetch_gnews(from_date, to_date):
    if not GNEWS_KEY:
        print("Skipping GNews: missing key")
        return []
    rows = []
    endpoint = "https://gnews.io/api/v4/search"
    start_d = date.fromisoformat(from_date)
    end_d   = date.fromisoformat(to_date)
    for query in QUERY_TERMS:
        params = {
            "q": query,
            "from": from_date + "T00:00:00Z",
            "to": to_date + "T23:59:59Z",
            "lang": "en",
            "max": 10,
            "token": GNEWS_KEY,
        }
        try:
            resp = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
        except requests.RequestException as e:
            print(f"GNews request failed: {e}")
            continue
        if resp.status_code != 200:
            print(f"GNews error {resp.status_code}: {resp.text[:120]}")
            continue
        for item in resp.json().get("articles", []):
            row = make_row(
                source=item.get("source", {}).get("name") or "GNews",
                published_iso=item.get("publishedAt", ""),
                title=item.get("title", ""),
                description=item.get("description", ""),
                url=item.get("url", ""),
                query=query,
                start_date=start_d,
                end_date=end_d,
            )
            if row:
                rows.append(row)
        print(f"GNews | {query} | rows {len(rows)}")
        time.sleep(0.5)
    return rows

# ----------------------------------------------------------------------
# 5. Deduplication
# ----------------------------------------------------------------------
def deduplicate_csv(csv_path):
    """Read CSV, drop duplicates by URL and Date+Headline, save sorted."""
    if not csv_path.exists():
        return

    # FIX: on_bad_lines='skip' drops malformed rows instead of crashing.
    # dtype=str prevents pandas from misinterpreting mixed-type columns.
    try:
        df = pd.read_csv(
            csv_path,
            on_bad_lines='skip',
            quoting=csv.QUOTE_ALL,
            dtype=str,
        )
    except Exception:
        # Fallback: try without QUOTE_ALL in case the file used default quoting
        try:
            df = pd.read_csv(csv_path, on_bad_lines='skip', dtype=str)
        except Exception as e:
            print(f"Could not read CSV for deduplication: {e}")
            return

    if df.empty:
        return

    # FIX: Remove spurious header rows that get written during repeated appends
    df = df[df["Archive"] != "Archive"]

    df["__url_key"] = df["Headline link"].astype(str).str.strip().str.lower()
    df["__title_key"] = df["Date"].astype(str) + "|" + df["Headline"].astype(str).str.lower().str.strip()
    df = df.drop_duplicates(subset=["__url_key"], keep="last")
    df = df.drop_duplicates(subset=["__title_key"], keep="last")
    df = df.drop(columns=["__url_key", "__title_key"])
    df["__sort_date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")
    df = df.sort_values(["__sort_date", "Source", "Headline"])
    df = df.drop(columns=["__sort_date"])

    # Write back using QUOTE_ALL so commas inside fields are always safe
    df.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)
    print(f"Deduplicated CSV. Total rows: {len(df)}")

# ----------------------------------------------------------------------
# 6. Main routine
# ----------------------------------------------------------------------
def main():
    last_run = get_last_run_date()
    today = date.today()

    if last_run >= today:
        print(f"Already up to date (last run = {last_run}). Nothing to do.")
        return

    from_date = last_run.isoformat()
    to_date   = today.isoformat()

    print(f"Fetching news from {from_date} to {to_date}...")

    all_rows = []
    all_rows.extend(fetch_guardian(from_date, to_date))
    all_rows.extend(fetch_newsapi(from_date, to_date))
    all_rows.extend(fetch_gnews(from_date, to_date))

    print(f"Total raw articles fetched: {len(all_rows)}")

    # Filter by impact score
    impact_rows = [r for r in all_rows if r.get("impact_score", 0) >= MIN_IMPACT_SCORE]
    print(f"Articles passing impact filter (score >= {MIN_IMPACT_SCORE}): {len(impact_rows)}")

    if not impact_rows:
        print("No new impact articles to save.")
        set_last_run_date(today)
        return

    df_new = pd.DataFrame(impact_rows)

    # Ensure all expected columns exist (for compatibility with old CSV)
    base_cols = ["Archive", "Date", "Headline", "Description", "Headline link", "Source", "Query"]
    for col in base_cols:
        if col not in df_new.columns:
            df_new[col] = ""

    # FIX: Use quoting=csv.QUOTE_ALL on both write paths so that commas/quotes
    # inside headlines or descriptions never break the CSV structure.
    if OUTPUT_FILE.exists():
        df_new.to_csv(
            OUTPUT_FILE,
            mode='a',
            header=False,
            index=False,
            quoting=csv.QUOTE_ALL,
        )
    else:
        df_new.to_csv(OUTPUT_FILE, index=False, quoting=csv.QUOTE_ALL)

    print(f"Appended {len(df_new)} rows to {OUTPUT_FILE}")

    # Clean duplicates from the whole file
    deduplicate_csv(OUTPUT_FILE)

    # Update last run date
    set_last_run_date(today)
    print(f"Last run date updated to {today}")

# ----------------------------------------------------------------------
# 7. Entry point
# ----------------------------------------------------------------------
if __name__ == "__main__":
    main()
else:
    print("Script loaded. In Jupyter, run main() to fetch new impact news.")